# Metric 2 — COMET QE Score
Run from project root: `jupyter notebook evaluation/visualisations/metric_2_comet.ipynb`

In [ ]:
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.dpi'] = 120

METRICS_DIR = Path('../../eval_results/metrics')
PATH = METRICS_DIR / 'metric_2.jsonl'

CONFIGS = [
    'api_no_glos', 'api_glos',
    'local_no_glos', 'local_glos',
    'finetuned_no_glos', 'finetuned_glos',
]
LABELS = {
    'api_no_glos':       'API\n(no glos)',
    'api_glos':          'API\n(glos)',
    'local_no_glos':     'Local\n(no glos)',
    'local_glos':        'Local\n(glos)',
    'finetuned_no_glos': 'Finetuned\n(no glos)',
    'finetuned_glos':    'Finetuned\n(glos)',
}

prob_rows, sol_rows = [], []
with open(PATH) as f:
    for line in f:
        rec = json.loads(line)
        idx = rec['idx']
        prob_rows.append({**{'idx': idx}, **{ck: (rec[ck]['problem'] if rec.get(ck) else None) for ck in CONFIGS}})
        sol_rows.append( {**{'idx': idx}, **{ck: (rec[ck]['solution'] if rec.get(ck) else None) for ck in CONFIGS}})

df_prob = pd.DataFrame(prob_rows).set_index('idx')
df_sol  = pd.DataFrame(sol_rows).set_index('idx')
print(f'Loaded {len(df_prob)} examples')
df_prob.describe()

In [ ]:
# --- Grouped bar: avg problem vs solution per config ---
prob_avgs = df_prob[CONFIGS].mean()
sol_avgs  = df_sol[CONFIGS].mean()

x = np.arange(len(CONFIGS))
w = 0.35
fig, ax = plt.subplots(figsize=(12, 5))
b1 = ax.bar(x - w/2, prob_avgs, w, label='Problem', color='steelblue')
b2 = ax.bar(x + w/2, sol_avgs,  w, label='Solution', color='coral')
ax.bar_label(b1, fmt='%.3f', padding=3, fontsize=8)
ax.bar_label(b2, fmt='%.3f', padding=3, fontsize=8)
ax.set_xticks(x)
ax.set_xticklabels([LABELS[c] for c in CONFIGS])
ax.set_ylim(0, 1.1)
ax.set_ylabel('COMET score')
ax.set_title('Metric 2 — COMET QE: Problem vs Solution')
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# --- Box plots: score distribution per config ---
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for ax, df, title in zip(axes, [df_prob, df_sol], ['Problems', 'Solutions']):
    data   = [df[c].dropna().values for c in CONFIGS]
    labels = [LABELS[c].replace('\n', ' ') for c in CONFIGS]
    bp = ax.boxplot(data, labels=labels, patch_artist=True,
                    boxprops=dict(facecolor='steelblue', alpha=0.6))
    ax.set_ylim(0, 1.05)
    ax.set_ylabel('COMET score')
    ax.set_title(f'Distribution — {title}')
    plt.setp(ax.get_xticklabels(), rotation=20, ha='right', fontsize=8)
plt.suptitle('Metric 2 — COMET Score Distribution', y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# --- Heatmap: model x glossary ---
models = ['api', 'local', 'finetuned']
for title, df in [('Problems', df_prob), ('Solutions', df_sol)]:
    heat = pd.DataFrame({
        'No glossary': [df[f'{m}_no_glos'].mean() for m in models],
        'Glossary':    [df[f'{m}_glos'].mean()    for m in models],
    }, index=['API', 'Local', 'Finetuned'])

    fig, ax = plt.subplots(figsize=(6, 3))
    sns.heatmap(heat, annot=True, fmt='.3f', cmap='YlGn', vmin=0, vmax=1,
                linewidths=0.5, ax=ax)
    ax.set_title(f'Metric 2 — COMET: {title}')
    plt.tight_layout()
    plt.show()

In [ ]:
# --- Scatter: problem score vs solution score per config ---
fig, axes = plt.subplots(2, 3, figsize=(13, 8))
for ax, ck in zip(axes.flat, CONFIGS):
    x = df_prob[ck].dropna()
    y = df_sol[ck].dropna()
    common = x.index.intersection(y.index)
    ax.scatter(x[common], y[common], alpha=0.4, s=15)
    lims = [min(x.min(), y.min()) - 0.02, max(x.max(), y.max()) + 0.02]
    ax.plot(lims, lims, 'r--', linewidth=0.8, label='y=x')
    ax.set_xlabel('Problem score')
    ax.set_ylabel('Solution score')
    ax.set_title(LABELS[ck].replace('\n', ' '))
plt.suptitle('Metric 2 — Problem vs Solution COMET score per example', y=1.01)
plt.tight_layout()
plt.show()